In [1]:
include("./Envs/Env.jl")
# include("./Algorithms/Qlearning.jl")
# include("./Algorithms/DQN.jl")
include("./Algorithms/DRQN.jl")

create_lstm_agent (generic function with 1 method)

In [2]:
using RockSample
# pomdp = RockSamplePOMDP(20, 20)
# pomdp_name = "RS2020"
# pomdp = RockSamplePOMDP(15, 15)
# pomdp_name = "RS1515"
# pomdp = RockSamplePOMDP(11, 11)
# pomdp_name = "RS1111"
# pomdp = RockSamplePOMDP(7, 8)
# pomdp_name = "RS78"
pomdp = RockSamplePOMDP(4, 3)
pomdp_name = "RS43"
bool_full_observability = false
env = Env(pomdp, bool_full_observability)
action_space = GetActionSpace(env)

1:8

In [3]:
# define convert_o function
function POMDPs.convert_o(T::Type{<:AbstractArray}, o::Int64, m::RockSamplePOMDP)
    vec = zeros(Float32, 3)
    vec[o] = 1.0f0
    return vec
end

In [4]:
# train(agent_Q_learning, training_episodes, env)

In [5]:
# evaluate_policy(agent_Q_learning, env, evaluation_episodes, max_steps)

In [6]:
# state_size = length(GetStateSpace(env))
state_dim = GetObsDim(env)
layer_size = 64
rnn_hidden_size = 64
lr = 5e-5 #learning rate
gamma = discount(pomdp)
start_exploration_rate = 1.0
min_exploration_rate = 0.3
training_episodes = 100000
# exploration_decay = 0.99999
exploration_decay = exp(log(min_exploration_rate / start_exploration_rate) / training_episodes)
# num_episodes = 50000
max_steps = 100
slide_window = 1000
batch_size = 4

RSState{3}([1, 1], Bool[1, 0, 0])
Float32[0.0, 1.0, 0.0]


4

In [7]:
state_dim

3

In [8]:
action_space

1:8

In [9]:
include("./Algorithms/DRQN.jl")
# agent = create_lstm_agent(action_space, state_dim;lr=lr, hidden_dim=layer_size, batch_size=batch_size, ϵ_end=0.1f0, ϵ_decay=0.99995, target_update_freq=1000, device=Flux.cpu) 
agent = create_lstm_agent(action_space, state_dim; γ = 0.999, lr=lr, hidden_dim=layer_size, rnn_hidden_size=rnn_hidden_size, batch_size=batch_size, ϵ_end=0.05f0, ϵ_decay=0.9995f0, device=Flux.cpu) 

DRQNAgent(Chain(Dense(11 => 11, tanh), LSTM(11 => 64), Dense(64 => 64, tanh), Dense(64 => 8)), Chain(Dense(11 => 11, tanh), LSTM(11 => 64), Dense(64 => 64, tanh), Dense(64 => 8)), (layers = ((weight = Leaf(Adam(eta=5.0e-5, beta=(0.9, 0.999), epsilon=1.0e-8), (Float32[0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; … ; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0], Float32[0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; … ; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0], (0.9, 0.999))), bias = Leaf(Adam(eta=5.0e-5, beta=(0.9, 0.999), epsilon=1.0e-8), (Float32[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], Float32[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], (0.9, 0.999))), σ = ()), (cell = (Wi = Leaf(Adam(eta=5.0e-5, beta=(0.9, 0.999), epsilon=1.0e-8), (Float32[0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; … ; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0], Float32[0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; … ; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0], (0.9, 0.999))), Wh = Leaf(Adam(eta=5.0e-5, beta=(0.9, 0.999), epsilon=1.0e-8

In [10]:
# 训练
rewards, losses, evals = train!(env, agent, training_episodes; max_steps=max_steps)
# rewards, losses, evals = train!(agent, env; episodes=num_episodes, max_steps=max_steps)

Progress:   0%|█                                        |  ETA: 1 days, 13:22:27┌ Warning: `Flux.params(m...)` is deprecated. Use `Flux.trainable(model)` for parameter collection,
│ and the explicit `gradient(m -> loss(m, x, y), model)` for gradient computation.
└ @ Flux C:\Users\MECHREVO\.julia\packages\Flux\BkG8S\src\deprecations.jl:93
Progress:   0%|█                                        |  ETA: 15:35:0347m

Episode 100 | Avg Reward: 3.7 | Avg Loss: 2.9226 | Eval Score: 10.0 | ϵ: 0.951 | Buffer: 59 episodes


Progress:   0%|█                                        |  ETA: 8:58:42m

Episode 200 | Avg Reward: 0.5 | Avg Loss: 2.9367 | Eval Score: 8.0 | ϵ: 0.905 | Buffer: 120 episodes


Progress:   0%|█                                        |  ETA: 6:40:31

...

Progress:  99%|█████████████████████████████████████████|  ETA: 0:00:09

Episode 99900 | Avg Reward: 0.1 | Avg Loss: 0.0864 | Eval Score: 0.0 | ϵ: 0.05 | Buffer: 10000 episodes


Progress: 100%|█████████████████████████████████████████| Time: 2:33:16


Episode 100000 | Avg Reward: -0.2 | Avg Loss: 0.0322 | Eval Score: 0.0 | ϵ: 0.05 | Buffer: 10000 episodes


(Float32[0.0, 10.0, -20.0, 10.0, 10.0, 0.0, 0.0, 10.0, 10.0, 10.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], Float32[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.9103572, 3.1190042, 3.1146474, 2.3559556  …  8.677275f-5, 7.454501f-5, 8.213436f-5, 9.821051f-5, 0.00012836396, 8.570075f-5, 0.00012001085, 0.0001294524, 0.0004902668, 0.00016705233], Float32[10.0, 8.0, 10.0, 8.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0])

In [11]:
evaluate(env, agent; num_episodes=1000, max_steps=max_steps)  # Replace with the updated evaluation function

0.0f0

In [12]:
outname = pomdp_name * "SavedModel.jld2"

"RS43SavedModel.jld2"

In [13]:
SaveModel(agent, env, outname)

LoadError: UndefVarError: `SaveModel` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [14]:
@load outname saved_model

LoadError: LoadError: UndefVarError: `@load` not defined in `Main`
Suggestion: check for spelling errors or missing imports.
in expression starting at In[14]:1

In [15]:
@time evaluate_cpu(saved_model; num_episodes=1000, max_steps=100)

LoadError: UndefVarError: `evaluate_cpu` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [16]:
@time evaluate(env, agent; num_episodes=1000, max_steps=100)  # Replace with the updated evaluation function

  1.281850 seconds (10.03 M allocations: 860.256 MiB, 9.54% gc time)


0.0f0

In [17]:
pomdp

RockSamplePOMDP{3}
  map_size: Tuple{Int64, Int64}
  rocks_positions: StaticArraysCore.SVector{3, StaticArraysCore.SVector{2, Int64}}
  init_pos: StaticArraysCore.SVector{2, Int64}
  sensor_efficiency: Float64 20.0
  bad_rock_penalty: Float64 -10.0
  good_rock_reward: Float64 10.0
  step_penalty: Float64 0.0
  sensor_use_penalty: Float64 0.0
  exit_reward: Float64 10.0
  terminal_state: RSState{3}
  indices: Array{Int64}((4,)) [4, 16, 32, 64]
  discount_factor: Float64 0.95


In [18]:
initialobs(pomdp)

LoadError: MethodError: no method matching initialobs(::RockSamplePOMDP{3})
The function `initialobs` exists, but no method is defined for this combination of argument types.

[0mClosest candidates are:
[0m  initialobs([91m::TigerPOMDP[39m, [91m::Bool[39m)
[0m[90m   @[39m [35mPOMDPModels[39m [90mC:\Users\MECHREVO\.julia\packages\POMDPModels\7tm5C\src\[39m[90m[4mTigerPOMDPs.jl:79[24m[39m
[0m  initialobs([91m::BabyPOMDP[39m, [91m::Any[39m)
[0m[90m   @[39m [35mPOMDPModels[39m [90mC:\Users\MECHREVO\.julia\packages\POMDPModels\7tm5C\src\[39m[90m[4mCryingBabies.jl:29[24m[39m
[0m  initialobs([91m::TMaze[39m, [91m::Any[39m)
[0m[90m   @[39m [35mPOMDPModels[39m [90mC:\Users\MECHREVO\.julia\packages\POMDPModels\7tm5C\src\[39m[90m[4mTMazes.jl:96[24m[39m
[0m  ...
